# Homework 4 — Colab 一鍵流程

**執行順序**：由上到下跑完所有 Code 儲存格即可（GPU：執行階段 → 變更執行階段類型 → T4）。

一鍵開 Colab（推上 `main` 後）：[在 Colab 開啟此 notebook](https://colab.research.google.com/github/alanhdchu154/adl_hw4/blob/main/notebooks/colab_full_pipeline.ipynb)

1. 填下面第一格常數（GitHub URL、UT ID）。
2. 依序執行：clone → 依賴 → 資料 → 標註 → VLM → CLIP → bundle + grader。

預設為 **smoke（極短訓練）**：只確認能產 checkpoint、能 bundle、能 grader。要認真拉分請把第一格裡 `VLM_*` / `CLIP_*` 改成下面「正式訓練建議」的數字。

若專案已經在 Colab 的 `/content` 裡（例如上傳 zip），可改 `SKIP_CLONE = True` 並把 `PROJECT_ROOT` 指到解壓後的目錄。

In [ ]:
# ========= 只改這裡 =========
REPO_URL = "https://github.com/alanhdchu154/adl_hw4.git"
CLONE_DIR = "adl_hw4"                # git clone 預設資料夾名 = repo 名稱
UTID = "ahc982"                      # bundle / zip 檔名（你的 UT ID）
SKIP_CLONE = False                   # 若已在 /content 有專案，改 True 並設定 PROJECT_ROOT
PROJECT_ROOT = f"/content/{CLONE_DIR}"  # 專案根目錄（內含 homework/, grader/, bundle.py）

# --- smoke（幾分鐘內跑完，驗流程）---
VLM_MAX_SAMPLES = 4096
VLM_MAX_STEPS = 30
CLIP_MAX_SAMPLES = 4096
CLIP_MAX_STEPS = 30
VLM_BATCH = 4
VLM_GRAD_ACCUM = 4
CLIP_BATCH = 16
# --- 正式訓練可改（範例，較久）---
# VLM_MAX_SAMPLES, VLM_MAX_STEPS = 80000, 400
# CLIP_MAX_SAMPLES, CLIP_MAX_STEPS = 80000, 400
# ===========================

import os
import subprocess
from pathlib import Path

if not SKIP_CLONE:
    if Path(PROJECT_ROOT).exists():
        print("目錄已存在，改為 git pull")
        subprocess.run(["git", "-C", PROJECT_ROOT, "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_ROOT], check=True)
else:
    assert Path(PROJECT_ROOT).exists(), f"找不到 {PROJECT_ROOT}，請檢查 SKIP_CLONE / PROJECT_ROOT"

os.chdir(PROJECT_ROOT)
print("工作目錄:", os.getcwd())

In [ ]:
import os
os.chdir(PROJECT_ROOT)

!pip install -q -r requirements.txt
%env HF_HUB_DISABLE_TELEMETRY=1
%env TOKENIZERS_PARALLELISM=false

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
os.chdir(PROJECT_ROOT)

DATA_URL = "https://utexas.box.com/shared/static/qubjm5isldqvyimfj9rsmbnvnbezwcv4.zip"
ZIP_NAME = "supertux_data.zip"

if not os.path.isfile(ZIP_NAME):
    !wget -q "{DATA_URL}" -O {ZIP_NAME}
else:
    print("已存在", ZIP_NAME)

!unzip -q -o {ZIP_NAME}
print("解壓完成。檢查:", os.path.isdir("data/train"), "train 圖張數範例:")
!ls data/train/*_im.jpg 2>/dev/null | head -3

In [ ]:
import os
os.chdir(PROJECT_ROOT)

!python -m homework.generate_qa build_train_qa
!python -m homework.generate_captions build_train_captions

In [ ]:
import os
os.chdir(PROJECT_ROOT)

!python -m homework.finetune train --output_dir vlm_model \
  --max_train_samples {VLM_MAX_SAMPLES} --max_steps {VLM_MAX_STEPS} --num_train_epochs 1 \
  --per_device_train_batch_size {VLM_BATCH} --gradient_accumulation_steps {VLM_GRAD_ACCUM} \
  --num_workers 2 --learning_rate 4e-4

In [ ]:
import os
os.chdir(PROJECT_ROOT)

!python -m homework.clip train --output_dir clip_model \
  --max_train_samples {CLIP_MAX_SAMPLES} --max_steps {CLIP_MAX_STEPS} --num_train_epochs 1 \
  --per_device_train_batch_size {CLIP_BATCH} --gradient_accumulation_steps 1 \
  --num_workers 2 --learning_rate 4e-4

In [ ]:
import os
os.chdir(PROJECT_ROOT)

# 本機驗證（可略過；若要省時間可註解掉）
!python -m homework.finetune test vlm_model || true
!python -m homework.clip test clip_model || true

In [ ]:
import os
os.chdir(PROJECT_ROOT)

!python3 bundle.py homework {UTID}
!python3 -m grader {UTID}.zip

## 下載交作業用 zip

上一格跑完後，左側檔案瀏覽器在專案根目錄會看到 `{UTID}.zip`，右鍵下載即可。若 zip 超過約 50MB，刪除 `homework/vlm_model/tensorboard`、`homework/clip_model/tensorboard` 後再跑一次 bundle 儲存格。

In [ ]:
from google.colab import files
import os
os.chdir(PROJECT_ROOT)
zp = f"{UTID}.zip"
if os.path.isfile(zp):
    files.download(zp)
else:
    print("找不到", zp, "請先執行 bundle 那一格")